# AICU LoRA Trainer **LTS-2026.06**
### 「画像生成AI Stable Diffusion スタートガイド」サポートノートブック

Base UX: [Lora Trainer by Hollowstrawberry](https://github.com/hollowstrawberry/kohya-colab) / Engine: [kohya-ss/sd-scripts **v0.10.5**](https://github.com/kohya-ss/sd-scripts) (tag固定)

✅ **ランタイム再起動は不要です**（numpy をダウングレードしない設計）
✅ Civitai モデル対応（Colab Secrets の `CIVITAI_KEY` を使用）
✅ SD1.5 / SDXL 両対応（自動でトレーニングスクリプトを切替）

メンテナンス: [AICU Inc.](https://corp.aicu.ai/) | 問い合わせ: X [@AICUai](https://x.com/AICUai)

### ⭕ Disclaimer 【免責事項】
本ノートブックは教育目的で提供されています。生成物のライセンスは使用するベースモデルとデータセットのライセンスに従ってください。
Civitai からダウンロードするモデルは各モデルページのライセンスに同意した上でご利用ください。

In [ ]:
#@title 🧪 QA parameters (papermill用 / 通常は触らない)
# このセルは papermill の parameters タグ付きセルです。
# CLI QA時に SMOKE_TEST=True が注入されます。
SMOKE_TEST = False


In [ ]:
#@title 📂 1. Google Drive に接続
import os, sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Not on Colab — skipping Drive mount (QA mode)")


In [ ]:
#@title 🔑 2. CIVITAI_KEY を読み込み（Civitaiモデルを使う場合のみ）
#@markdown Colab 左側の 🔑 Secrets パネルに `CIVITAI_KEY` を登録し、
#@markdown このノートブックへのアクセスを許可してください。
#@markdown APIキーの取得: https://civitai.com/user/account
import sys
CIVITAI_KEY = None
if "google.colab" in sys.modules:
    from google.colab import userdata
    try:
        CIVITAI_KEY = userdata.get("CIVITAI_KEY")
    except Exception:
        CIVITAI_KEY = None
if CIVITAI_KEY:
    print("✅ CIVITAI_KEY: loaded from Colab Secrets")
else:
    print("ℹ️ CIVITAI_KEY なし。HuggingFace モデルのみ利用可能です。")


In [ ]:
#@markdown ### 📦 3. サンプルデータセットの準備（任意）
#@markdown マイドライブの「Loras」フォルダに[サンプルデータセット](https://huggingface.co/AICU/SDXL-LoRA/tree/main)をダウンロードして展開します。
import os, sys

download_from = "https://huggingface.co/AICU/SDXL-LoRA/resolve/main/9shoku0219.zip" #@param  {type:"string"}
zip_path = "/content/drive/MyDrive/Loras/9shoku0219.zip" #@param {type:"string"}
extract_to = "/content/drive/MyDrive/Loras/9shoku0219/dataset" #@param {type:"string"}

if "google.colab" in sys.modules:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        print("📂 Google Drive に接続中...")
        drive.mount('/content/drive')
    os.makedirs(os.path.dirname(zip_path), exist_ok=True)
    os.makedirs(extract_to, exist_ok=True)
    get_ipython().system(f"wget -nc '{download_from}' -O '{zip_path}'")
    get_ipython().system(f"unzip -n -j '{zip_path}' -d '{extract_to}'")
else:
    print("Not on Colab — skipping dataset download (QA mode)")


## 🚩 4. トレーニング本体
下のフォームを設定して実行してください。**このセルだけで 依存インストール → モデルDL → 学習 まで完結します。**
初回実行時のインストールは 4〜7 分程度です。**再起動は不要です。**

In [ ]:
import os
import re
import sys
import toml
from time import time

# ============================================================
# AICU LoRA Trainer LTS-2026.06
# Base UX: Lora Trainer by Hollowstrawberry
# Stack:   kohya-ss/sd-scripts v0.10.5 (tag-pinned)
# Policy:  No numpy downgrade -> NO RUNTIME RESTART REQUIRED
# ============================================================

# --- LTS pinned stack (DO NOT change casually; see README) ---
KOHYA_REPO  = "https://github.com/kohya-ss/sd-scripts.git"
KOHYA_REF   = "v0.10.5"                  # tag pin = reproducible
TORCH_VER   = "2.6.0"
TVISION_VER = "0.21.0"
XFORMERS_VER= "0.0.29.post3"             # built for torch 2.6.0
CUDA_INDEX  = "https://download.pytorch.org/whl/cu124"
BNB_VER     = "0.45.5"                   # AdamW8bit, numpy2-OK

COLAB = "google.colab" in sys.modules or os.path.exists("/content")
SMOKE_TEST = bool(globals().get("SMOKE_TEST", False) or os.environ.get("SMOKE_TEST"))
LOAD_TRUNCATED_IMAGES = True

# These carry information from past executions
if "model_url" in globals():
  old_model_url = model_url
else:
  old_model_url = None
if "dependencies_installed" not in globals():
  dependencies_installed = False
if "model_file" not in globals():
  model_file = None

# These may be set by other cells, some are legacy
if "custom_dataset" not in globals():
  custom_dataset = None
if "override_dataset_config_file" not in globals():
  override_dataset_config_file = None
if "override_config_file" not in globals():
  override_config_file = None
if "optimizer" not in globals():
  optimizer = "AdamW8bit"
if "optimizer_args" not in globals():
  optimizer_args = None
if "continue_from_lora" not in globals():
  continue_from_lora = ""
if "weighted_captions" not in globals():
  weighted_captions = False
if "adjust_tags" not in globals():
  adjust_tags = False
if "keep_tokens_weight" not in globals():
  keep_tokens_weight = 1.0

#@title ## 🚩 Start Here

#@markdown ### ▶️ Setup
#@markdown Your project name will be the same as the folder containing your images. Spaces aren't allowed.
#@markdown プロジェクト名は、画像を含むフォルダーと同じになります。スペースは使用できません。
project_name = "9shoku0219" #@param {type:"string"}
#@markdown The folder structure doesn't matter and is purely for comfort. Make sure to always pick the same one.
#@markdown フォルダー構造は重要ではなく、単に利便性のためです。常に同じものを選択するようにしてください。
folder_structure = "Organize by project (MyDrive/Loras/project_name/dataset)" #@param ["Organize by category (MyDrive/lora_training/datasets/project_name)", "Organize by project (MyDrive/Loras/project_name/dataset)"]
#@markdown Choose the base model. Civitai links are supported (requires `CIVITAI_KEY` in Colab Secrets — run the 🔑 cell above).
#@markdown ベースモデルを選択します。Civitai リンク対応（Colab Secrets に `CIVITAI_KEY` が必要です — 上の 🔑 セルを実行してください）。
training_model = "AnyLora (AnyLoRA_noVae_fp16-pruned.ckpt)" #@param ["Anime (animefull-final-pruned-fp16.safetensors)", "AnyLora (AnyLoRA_noVae_fp16-pruned.ckpt)", "Stable Diffusion (sd-v1-5-pruned-noema-fp16.safetensors)", "Sierunami v1 (Civitai, SDXL)", "Animagine XL 4.0 (SDXL)"]
#@markdown Custom URL: HuggingFace direct/blob links, Civitai model pages with `?modelVersionId=`, or `civitai.com/api/download/models/...` links.
optional_custom_training_model_url = "" #@param {type:"string"}
custom_model_is_based_on_sd2 = False #@param {type:"boolean"}
#@markdown Check this if your custom model is SDXL (switches to sdxl_train_network.py, resolution 1024 recommended).
#@markdown カスタムモデルが SDXL の場合はチェック（sdxl_train_network.py に切替。解像度 1024 推奨）。
custom_model_is_sdxl = False #@param {type:"boolean"}

is_sdxl = custom_model_is_sdxl
if optional_custom_training_model_url:
  model_url = optional_custom_training_model_url
elif "Sierunami" in training_model:
  model_url = "https://civitai.com/api/download/models/1176276"
  is_sdxl = True
elif "Animagine" in training_model:
  model_url = "https://huggingface.co/cagliostrolab/animagine-xl-4.0/resolve/main/animagine-xl-4.0.safetensors"
  is_sdxl = True
elif "AnyLora" in training_model:
  model_url = "https://huggingface.co/Lykon/AnyLoRA/resolve/main/AnyLoRA_noVae_fp16-pruned.ckpt"
elif "Anime" in training_model:
  model_url = "https://huggingface.co/hollowstrawberry/stable-diffusion-guide/resolve/main/models/animefull-final-pruned-fp16.safetensors"
else:
  model_url = "https://huggingface.co/hollowstrawberry/stable-diffusion-guide/resolve/main/models/sd-v1-5-pruned-noema-fp16.safetensors"

#@markdown ### ▶️ Processing
#@markdown Resolution 512 is standard for SD1.5. Use 1024 for SDXL models.
#@markdown 解像度 512 は SD1.5 の標準です。SDXL モデルでは 1024 を使用してください。
resolution = 512 #@param {type:"slider", min:512, max:1024, step:128}
flip_aug = False #@param {type:"boolean"}
caption_extension = ".txt"
#@markdown Shuffling anime tags in place improves learning and prompting. An activation tag goes at the start of every text file and will not be shuffled.
#@markdown animeタグを所定の位置にシャッフルすると、学習とプロンプトが向上します。activationタグは各テキストファイルの先頭に配置します(シャッフルされません)。
shuffle_tags = True #@param {type:"boolean"}
shuffle_caption = shuffle_tags
activation_tags = "1" #@param [0,1,2,3]
keep_tokens = int(activation_tags)

#@markdown ### ▶️ Steps
#@markdown 画像の数 × 繰り返し回数は 200〜400 を推奨します。
num_repeats = 10 #@param {type:"number"}
#@markdown 約 10 エポックまたは約 2000 ステップが適切な開始点です。
preferred_unit = "Epochs" #@param ["Epochs", "Steps"]
how_many = 10 #@param {type:"number"}
max_train_epochs = how_many if preferred_unit == "Epochs" else None
max_train_steps = how_many if preferred_unit == "Steps" else None
save_every_n_epochs = 1 #@param {type:"number"}
keep_only_last_n_epochs = 10 #@param {type:"number"}
if not save_every_n_epochs:
  save_every_n_epochs = max_train_epochs
if not keep_only_last_n_epochs:
  keep_only_last_n_epochs = max_train_epochs
#@markdown バッチサイズ 2 または 3 を推奨（SDXL は 1 推奨）。
train_batch_size = 2 #@param {type:"slider", min:1, max:8, step:1}

#@markdown ### ▶️ Learning
unet_lr = 5e-4 #@param {type:"number"}
text_encoder_lr = 1e-4 #@param {type:"number"}
lr_scheduler = "cosine_with_restarts" #@param ["constant", "cosine", "cosine_with_restarts", "constant_with_warmup", "linear", "polynomial"]
lr_scheduler_number = 3 #@param {type:"number"}
lr_scheduler_num_cycles = lr_scheduler_number if lr_scheduler == "cosine_with_restarts" else 0
lr_scheduler_power = lr_scheduler_number if lr_scheduler == "polynomial" else 0
lr_warmup_ratio = 0.05 #@param {type:"slider", min:0.0, max:0.5, step:0.01}
lr_warmup_steps = 0
min_snr_gamma = True #@param {type:"boolean"}
min_snr_gamma_value = 5.0 if min_snr_gamma else None

#@markdown ### ▶️ Structure
lora_type = "LoRA" #@param ["LoRA", "LoCon"]
network_dim = 16 #@param {type:"slider", min:1, max:128, step:1}
network_alpha = 8 #@param {type:"slider", min:1, max:128, step:1}
conv_dim = 8 #@param {type:"slider", min:1, max:64, step:1}
conv_alpha = 4 #@param {type:"slider", min:1, max:64, step:1}

network_module = "networks.lora"
network_args = None
if lora_type.lower() == "locon":
  network_args = [f"conv_dim={conv_dim}", f"conv_alpha={conv_alpha}"]

#@markdown ### ▶️ Ready
#@markdown さあ このセルを実行して Lora ができあがります。グッドラック!

# 👩‍💻 Cool code goes here

if optimizer.lower() == "prodigy" or "dadapt" in optimizer.lower():
  if globals().get("override_values_for_dadapt_and_prodigy", False):
    unet_lr = 0.5
    text_encoder_lr = 0.5
    lr_scheduler = "constant_with_warmup"
    lr_warmup_ratio = 0.05
    network_alpha = network_dim
  if not optimizer_args:
    optimizer_args = ["decouple=True","weight_decay=0.01","betas=[0.9,0.999]"]
    if optimizer == "Prodigy":
      optimizer_args.extend(["d_coef=2","use_bias_correction=True"])
      optimizer_args.append("safeguard_warmup=True" if lr_warmup_ratio > 0 else "safeguard_warmup=False")

root_dir = "/content" if COLAB else os.path.abspath("./smoke_work")
repo_dir = os.path.join(root_dir, "sd-scripts")

if "/Loras" in folder_structure:
  main_dir      = os.path.join(root_dir, "drive/MyDrive/Loras") if COLAB else os.path.join(root_dir, "Loras")
  log_folder    = os.path.join(main_dir, "_logs")
  config_folder = os.path.join(main_dir, project_name)
  images_folder = os.path.join(main_dir, project_name, "dataset")
  output_folder = os.path.join(main_dir, project_name, "output")
else:
  main_dir      = os.path.join(root_dir, "drive/MyDrive/lora_training") if COLAB else os.path.join(root_dir, "lora_training")
  images_folder = os.path.join(main_dir, "datasets", project_name)
  output_folder = os.path.join(main_dir, "output", project_name)
  config_folder = os.path.join(main_dir, "config", project_name)
  log_folder    = os.path.join(main_dir, "log")

config_file = os.path.join(config_folder, "training_config.toml")
dataset_config_file = os.path.join(config_folder, "dataset_config.toml")
accelerate_config_file = os.path.join(repo_dir, "accelerate_config/config.yaml")


def get_civitai_key():
  """Colab Secrets または環境変数から CIVITAI_KEY を取得（なければ None）"""
  key = globals().get("CIVITAI_KEY") or os.environ.get("CIVITAI_KEY")
  if not key and COLAB:
    try:
      from google.colab import userdata
      key = userdata.get("CIVITAI_KEY")
    except Exception:
      key = None
  return key


def install_dependencies():
  os.chdir(root_dir)
  !git clone {KOHYA_REPO} {repo_dir}
  os.chdir(repo_dir)
  !git fetch --tags -q
  !git checkout -q {KOHYA_REF}

  !apt -y install -qq aria2 > /dev/null

  # --- LTS policy: install torch FIRST against the pinned CUDA index. ---
  # numpy is intentionally NOT pinned/downgraded: sd-scripts v0.10.5 is
  # numpy 2.x compatible, so Colab's preinstalled numpy stays untouched
  # and NO RUNTIME RESTART is required.
  !pip install -q torch=={TORCH_VER} torchvision=={TVISION_VER} --index-url {CUDA_INDEX}
  !pip install -q -r requirements.txt
  !pip install -q bitsandbytes=={BNB_VER}
  # xformers is optional; failure falls back to sdpa
  !pip install -q xformers=={XFORMERS_VER} --index-url {CUDA_INDEX} || echo "xformers unavailable, will use sdpa"

  if LOAD_TRUNCATED_IMAGES:
    !sed -i 's/from PIL import Image/from PIL import Image, ImageFile\nImageFile.LOAD_TRUNCATED_IMAGES=True/g' library/train_util.py

  from accelerate.utils import write_basic_config
  if not os.path.exists(accelerate_config_file):
    os.makedirs(os.path.dirname(accelerate_config_file), exist_ok=True)
    write_basic_config(save_location=accelerate_config_file)

  os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
  os.environ["BITSANDBYTES_NOWELCOME"] = "1"
  os.environ["SAFETENSORS_FAST_GPU"] = "1"


def check_no_restart_needed():
  """numpy の実体とインストール済みバージョンの整合を確認する。
  メジャーバージョンが変わっていなければ再起動不要。"""
  import importlib.metadata
  import numpy
  loaded = numpy.__version__
  installed = importlib.metadata.version("numpy")
  if loaded.split(".")[0] != installed.split(".")[0]:
    print(f"⚠️ numpy loaded={loaded} installed={installed} — メジャーバージョンが変わりました。")
    print("⚠️ ランタイム > セッションを再起動 を実行してから、このセルを再実行してください。")
    return False
  print(f"✅ numpy {loaded} (unchanged) — runtime restart NOT required.")
  return True


def validate_dataset():
  global lr_warmup_steps, lr_warmup_ratio, caption_extension, keep_tokens, keep_tokens_weight, weighted_captions, adjust_tags
  supported_types = (".png", ".jpg", ".jpeg", ".webp", ".bmp")

  print("\n💿 Checking dataset...")
  if not project_name.strip() or any(c in project_name for c in " .()\"'\\/"):
    print("💥 Error: Please choose a valid project name.")
    return

  if custom_dataset:
    try:
      datconf = toml.loads(custom_dataset)
      datasets = [d for d in datconf["datasets"][0]["subsets"]]
    except Exception:
      print("💥 Error: Your custom dataset is invalid or contains an error! Please check the original template.")
      return
    reg = [d.get("image_dir") for d in datasets if d.get("is_reg", False)]
    datasets_dict = {d["image_dir"]: d["num_repeats"] for d in datasets}
    folders = datasets_dict.keys()
    files = [f for folder in folders for f in os.listdir(folder)]
    images_repeats = {folder: (len([f for f in os.listdir(folder) if f.lower().endswith(supported_types)]), datasets_dict[folder]) for folder in folders}
  else:
    reg = []
    folders = [images_folder]
    files = os.listdir(images_folder)
    images_repeats = {images_folder: (len([f for f in files if f.lower().endswith(supported_types)]), num_repeats)}

  for folder in folders:
    if not os.path.exists(folder):
      print(f"💥 Error: The folder {folder.replace('/content/drive/', '')} doesn't exist.")
      return
  for folder, (img, rep) in images_repeats.items():
    if not img:
      print(f"💥 Error: Your {folder.replace('/content/drive/', '')} folder is empty.")
      return
  for f in files:
    if not f.lower().endswith((".txt", ".npz")) and not f.lower().endswith(supported_types):
      print(f"💥 Error: Invalid file in dataset: \"{f}\". Aborting.")
      return

  if not [txt for txt in files if txt.lower().endswith(".txt")]:
    caption_extension = ""
  if continue_from_lora and not (continue_from_lora.endswith(".safetensors") and os.path.exists(continue_from_lora)):
    print("💥 Error: Invalid path to existing Lora. Example: /content/drive/MyDrive/Loras/example.safetensors")
    return

  pre_steps_per_epoch = sum(img*rep for (img, rep) in images_repeats.values())
  steps_per_epoch = pre_steps_per_epoch/train_batch_size
  total_steps = max_train_steps or int(max_train_epochs*steps_per_epoch)
  estimated_epochs = int(total_steps/steps_per_epoch)
  lr_warmup_steps = int(total_steps*lr_warmup_ratio)

  for folder, (img, rep) in images_repeats.items():
    print("📁" + folder.replace("/content/drive/", "") + (" (Regularization)" if folder in reg else ""))
    print(f"📈 Found {img} images with {rep} repeats, equaling {img*rep} steps.")
  print(f"📉 Divide {pre_steps_per_epoch} steps by {train_batch_size} batch size to get {steps_per_epoch} steps per epoch.")
  if max_train_epochs:
    print(f"🔮 There will be {max_train_epochs} epochs, for around {total_steps} total training steps.")
  else:
    print(f"🔮 There will be {total_steps} steps, divided into {estimated_epochs} epochs and then some.")

  if total_steps > 10000:
    print("💥 Error: Your total steps are too high. You probably made a mistake. Aborting...")
    return

  if adjust_tags:
    print(f"\n📎 Weighted tags: {'ON' if weighted_captions else 'OFF'}")
    if weighted_captions:
      print(f"📎 Will use {keep_tokens_weight} weight on {keep_tokens} activation tag(s)")
    print("📎 Adjusting tags...")
    adjust_weighted_tags(folders, keep_tokens, keep_tokens_weight, weighted_captions)

  return True


def adjust_weighted_tags(folders, keep_tokens: int, keep_tokens_weight: float, weighted_captions: bool):
  weighted_tag = re.compile(r"\((.+?):[.\d]+\)(,|$)")
  for folder in folders:
    for txt in [f for f in os.listdir(folder) if f.lower().endswith(".txt")]:
      with open(os.path.join(folder, txt), 'r') as f:
        content = f.read()
      content = content.replace('\\', '')
      content = weighted_tag.sub(r'\1\2', content)
      if weighted_captions:
        content = content.replace(r'(', r'\(').replace(r')', r'\)').replace(r':', r'\:')
        if keep_tokens_weight > 1:
          tags = [s.strip() for s in content.split(",")]
          for i in range(min(keep_tokens, len(tags))):
            tags[i] = f'({tags[i]}:{keep_tokens_weight})'
          content = ", ".join(tags)
      with open(os.path.join(folder, txt), 'w') as f:
        f.write(content)


def create_config():
  global dataset_config_file, config_file, model_file

  if override_config_file:
    config_file = override_config_file
    print(f"\n⭕ Using custom config file {config_file}")
  else:
    config_dict = {
      "additional_network_arguments": {
        "unet_lr": unet_lr,
        "text_encoder_lr": text_encoder_lr,
        "network_dim": network_dim,
        "network_alpha": network_alpha,
        "network_module": network_module,
        "network_args": network_args,
        "network_train_unet_only": True if text_encoder_lr == 0 else None,
        "network_weights": continue_from_lora if continue_from_lora else None
      },
      "optimizer_arguments": {
        "learning_rate": unet_lr,
        "lr_scheduler": lr_scheduler,
        "lr_scheduler_num_cycles": lr_scheduler_num_cycles if lr_scheduler == "cosine_with_restarts" else None,
        "lr_scheduler_power": lr_scheduler_power if lr_scheduler == "polynomial" else None,
        "lr_warmup_steps": lr_warmup_steps if lr_scheduler != "constant" else None,
        "optimizer_type": optimizer,
        "optimizer_args": optimizer_args if optimizer_args else None,
      },
      "training_arguments": {
        "max_train_steps": max_train_steps,
        "max_train_epochs": max_train_epochs,
        "save_every_n_epochs": save_every_n_epochs,
        "save_last_n_epochs": keep_only_last_n_epochs,
        "train_batch_size": train_batch_size,
        "noise_offset": None,
        "clip_skip": None if is_sdxl else 2,
        "no_half_vae": True if is_sdxl else None,
        "min_snr_gamma": min_snr_gamma_value,
        "weighted_captions": weighted_captions,
        "seed": 42,
        "max_token_length": 225,
        "xformers": USE_XFORMERS if "USE_XFORMERS" in globals() else None,
        "sdpa": None if globals().get("USE_XFORMERS") else True,
        "lowram": COLAB,
        "max_data_loader_n_workers": 8,
        "persistent_data_loader_workers": True,
        "save_precision": "fp16",
        "mixed_precision": "fp16",
        "output_dir": output_folder,
        "logging_dir": log_folder,
        "output_name": project_name,
        "log_prefix": project_name,
      },
      "model_arguments": {
        "pretrained_model_name_or_path": model_file,
        "v2": custom_model_is_based_on_sd2,
        "v_parameterization": True if custom_model_is_based_on_sd2 else None,
      },
      "saving_arguments": {
        "save_model_as": "safetensors",
      },
      "dreambooth_arguments": {
        "prior_loss_weight": 1.0,
      },
      "dataset_arguments": {
        "cache_latents": True,
      },
    }

    for key in config_dict:
      if isinstance(config_dict[key], dict):
        config_dict[key] = {k: v for k, v in config_dict[key].items() if v is not None}

    with open(config_file, "w") as f:
      f.write(toml.dumps(config_dict))
    print(f"\n📄 Config saved to {config_file}")

  if override_dataset_config_file:
    dataset_config_file = override_dataset_config_file
    print(f"⭕ Using custom dataset config file {dataset_config_file}")
  else:
    dataset_config_dict = {
      "general": {
        "resolution": resolution,
        "shuffle_caption": shuffle_caption,
        "keep_tokens": keep_tokens,
        "flip_aug": flip_aug,
        "caption_extension": caption_extension,
        "enable_bucket": True,
        "bucket_reso_steps": 64,
        "bucket_no_upscale": False,
        "min_bucket_reso": 320 if resolution > 640 else 256,
        "max_bucket_reso": 1280 if resolution > 640 else 1024,
      },
      "datasets": toml.loads(custom_dataset)["datasets"] if custom_dataset else [
        {
          "subsets": [
            {
              "num_repeats": num_repeats,
              "image_dir": images_folder,
              "class_tokens": None if caption_extension else project_name
            }
          ]
        }
      ]
    }

    for key in dataset_config_dict:
      if isinstance(dataset_config_dict[key], dict):
        dataset_config_dict[key] = {k: v for k, v in dataset_config_dict[key].items() if v is not None}

    with open(dataset_config_file, "w") as f:
      f.write(toml.dumps(dataset_config_dict))
    print(f"📄 Dataset config saved to {dataset_config_file}")


def resolve_model_url(url):
  """URLを正規化し、(real_url, model_file, needs_civitai_key) を返す。
  対応: HuggingFace blob/resolve, Civitai モデルページ(?modelVersionId=), Civitai API直リンク"""
  real_url = url.strip()
  needs_key = False

  if real_url.lower().endswith((".ckpt", ".safetensors")):
    mfile = f"{root_dir}{real_url[real_url.rfind('/'):]}"
  else:
    mfile = f"{root_dir}/downloaded_model.safetensors"

  if re.search(r"(?:https?://)?(?:www\.)?huggingface\.co/[^/]+/[^/]+/blob", real_url):
    real_url = real_url.replace("blob", "resolve")
  elif re.search(r"civitai\.com/api/download/models/[0-9]+", real_url):
    needs_key = True
  elif m := re.search(r"(?:https?://)?(?:www\.)?civitai\.com/models/([0-9]+)(/[A-Za-z0-9-_]+)?", real_url):
    if m.group(2):
      mfile = f"{root_dir}{m.group(2)}.safetensors"
    if mv := re.search(r"modelVersionId=([0-9]+)", real_url):
      real_url = f"https://civitai.com/api/download/models/{mv.group(1)}"
      needs_key = True
    else:
      raise ValueError(
        "Civitai のモデルページ URL に modelVersionId が含まれていません。\n"
        "モデルページ右上のダウンロードボタンを右クリックしてリンクをコピーするか、\n"
        "URL 末尾に ?modelVersionId=XXXXXX を付けてください。")
  return real_url, mfile, needs_key


def download_model():
  global old_model_url, model_url, model_file
  real_model_url, model_file, needs_key = resolve_model_url(model_url)

  if os.path.exists(model_file) and not model_file.endswith("downloaded_model.safetensors"):
    pass
  elif os.path.exists(model_file):
    !rm "{model_file}"

  header_arg = ""
  if needs_key:
    civitai_key = get_civitai_key()
    if not civitai_key:
      print("💥 Error: Civitai モデルのダウンロードには CIVITAI_KEY が必要です。")
      print("   Colab 左側の 🔑 Secrets に CIVITAI_KEY を登録し、ノートブックへのアクセスを許可してください。")
      print("   APIキーの取得: https://civitai.com/user/account")
      return False
    header_arg = f'--header="Authorization: Bearer {civitai_key}"'

  !aria2c "{real_model_url}" {header_arg} --console-log-level=warn -c -s 16 -x 16 -k 10M -d / -o "{model_file}"

  # Civitai が HTML (ログイン/同意ページ) を返すケースの検出
  if os.path.exists(model_file) and os.path.getsize(model_file) < 1024 * 1024:
    with open(model_file, "rb") as f:
      head = f.read(512)
    if b"<html" in head.lower() or b"<!doctype" in head.lower():
      print("💥 Error: Civitai が HTML を返しました。CIVITAI_KEY の有効性とモデルのライセンス同意を確認してください。")
      return False

  if model_file.lower().endswith(".safetensors"):
    from safetensors.torch import load_file as load_safetensors
    try:
      test = load_safetensors(model_file)
      del test
    except Exception:
      new_model_file = os.path.splitext(model_file)[0] + ".ckpt"
      !mv "{model_file}" "{new_model_file}"
      model_file = new_model_file
      print(f"Renamed model to {model_file}")

  if model_file.lower().endswith(".ckpt"):
    from torch import load as load_ckpt
    try:
      test = load_ckpt(model_file)
      del test
    except Exception:
      return False

  return True


def main():
  global dependencies_installed, USE_XFORMERS, model_file

  if COLAB and not os.path.exists('/content/drive'):
    from google.colab import drive
    print("📂 Connecting to Google Drive...")
    drive.mount('/content/drive')

  for d in (main_dir, repo_dir, log_folder, images_folder, output_folder, config_folder):
    os.makedirs(d, exist_ok=True)

  if not validate_dataset():
    return

  if SMOKE_TEST:
    print("\n🧪 SMOKE_TEST: skipping install/download/training. Generating configs only.")
    model_file = model_file or "/dev/null"
    USE_XFORMERS = False
    create_config()
    print("\n🧪 SMOKE_TEST complete. Inspect the generated TOML files.")
    return

  if not dependencies_installed:
    print("\n🏭 Installing LTS dependencies (no numpy downgrade = no restart)...\n")
    t0 = time()
    install_dependencies()
    t1 = time()
    dependencies_installed = True
    print(f"\n✅ Installation finished in {int(t1-t0)} seconds.")
    if not check_no_restart_needed():
      return
  else:
    print("\n✅ Dependencies already installed.")

  try:
    import xformers  # noqa: F401
    USE_XFORMERS = True
  except Exception:
    USE_XFORMERS = False
    print("ℹ️ xformers not available — using sdpa attention.")

  if old_model_url != model_url or not model_file or not os.path.exists(model_file):
    print("\n🔄 Downloading model...")
    if not download_model():
      print("\n💥 Error: The model you selected is invalid, corrupted, or couldn't be downloaded.")
      return
    print()
  else:
    print("\n🔄 Model already downloaded.\n")

  create_config()

  train_script = "sdxl_train_network.py" if is_sdxl else "train_network.py"
  print(f"\n⭐ Starting trainer ({train_script}, {'SDXL' if is_sdxl else 'SD1.5'})...\n")
  os.chdir(repo_dir)

  !accelerate launch --config_file={accelerate_config_file} --num_cpu_threads_per_process=1 {train_script} --dataset_config={dataset_config_file} --config_file={config_file}

main()


## *️⃣ Extras

In [ ]:
#@markdown ### 🔮 Optimizer
#@markdown デフォルトは `AdamW8bit`（推奨）。Dadapt / Prodigy は学習率を自動管理し、小規模データセットに最適です。
optimizer = "AdamW8bit" #@param ["AdamW8bit", "Prodigy", "DAdaptation", "DadaptAdam", "DadaptLion", "AdamW", "Lion", "SGDNesterov", "SGDNesterov8bit", "AdaFactor"]
#@markdown Optional. 1行に1つの引数を `key=value` 形式で。
optimizer_args = "" #@param {type:"string"}
optimizer_args = [a.strip() for a in optimizer_args.split(",") if a.strip()] or None
override_values_for_dadapt_and_prodigy = True #@param {type:"boolean"}
print(f"Optimizer: {optimizer}")


### 📚 Multiple folders in dataset データセット内の複数のフォルダー
複数フォルダーや正則化画像を使う場合は、下のセルを編集して実行してから、トレーニング本体を実行してください。
使わない場合はその下の `custom_dataset = None` セルを実行します。

In [ ]:
# @title
custom_dataset = """
[[datasets]]

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/good_images"
num_repeats = 3

[[datasets.subsets]]
image_dir = "/content/drive/MyDrive/Loras/example/dataset/normal_images"
num_repeats = 1

"""


In [ ]:
custom_dataset = None


In [ ]:
#@markdown ### 🔢 Count datasets
folder = "/content/drive/MyDrive/Loras" #@param {type:"string"}
import os, sys

if "google.colab" in sys.modules and not os.path.exists('/content/drive'):
    from google.colab import drive
    print("📂 Connecting to Google Drive...\n")
    drive.mount('/content/drive')

tree = {}
exclude = ("_logs", "/output")
for i, (root, dirs, files) in enumerate(os.walk(folder, topdown=True)):
  dirs[:] = [d for d in dirs if all(ex not in d for ex in exclude)]
  images = len([f for f in files if f.lower().endswith((".png", ".jpg", ".jpeg"))])
  captions = len([f for f in files if f.lower().endswith(".txt")])
  others = len(files) - images - captions
  path = root[folder.rfind("/")+1:]
  tree[path] = None if not images else f"{images:>4} images | {captions:>4} captions |"
  if tree[path] and others:
    tree[path] += f" {others:>4} other files"

if tree and any(tree.values()):
  pad = max(len(k) for k in tree)
  print("\n".join(f"📁{k.ljust(pad)} | {v}" for k, v in tree.items() if v))
else:
  print("No images found.")


# 📈 Plot training results トレーニング結果をプロットする
TensorBoard でトレーニングの進行を確認できます。
⚠️ **保存時の注意**: TensorBoard 実行後にノートブックを保存すると、出力キャッシュで .ipynb が10MB以上肥大化します。保存前に「編集 > すべての出力を消去」を実行してください（QAスクリプト `clean_notebook.py` でも除去できます）。

In [ ]:
import sys
if "google.colab" in sys.modules:
    get_ipython().run_line_magic("load_ext", "tensorboard")
else:
    print("Not on Colab — skipping TensorBoard (QA mode)")


In [ ]:
# @title
import sys
log_folder = "/content/drive/MyDrive/Loras/_logs"
if "google.colab" in sys.modules:
    get_ipython().run_line_magic("tensorboard", f"--logdir {log_folder}")
else:
    print("Not on Colab — skipping TensorBoard (QA mode)")


## 9. Changelog
- **LTS-2026.06**: kohya-ss/sd-scripts v0.10.5 にエンジン移行。torch 2.6.0+cu124 / xformers 0.0.29.post3 / bitsandbytes 0.45.5 固定。numpy ダウングレード廃止により**ランタイム再起動が不要**に。Civitai API キー対応（Colab Secrets）。SDXL 対応（sdxl_train_network.py 自動切替）。TensorBoard キャッシュ肥大化の注意書きを追加。
- 旧版: uYouUs/sd-scripts + torch 2.4 cu121 + numpy 1.26.4（再起動必要）